# M3c Tier 1 QLoRA training: one model per session (PLAN §7 M3c; D-070, D-071, D-076)

**Notebook settings:** Accelerator **GPU T4 x2** · Internet **on** · Secrets: **HF_TOKEN**
(and **GITHUB_TOKEN** only if the repo is private). Run as **Save Version → Save & Run All**.

Set `MODEL` in the first code cell. Sessions, in order: `llama3.1-8b`, `mistral-7b`, `gemma3-4b`.
- **llama3.1-8b**: no probe. D-071 fixed batch 2×16 and 1 epoch (6.4 h projected).
- **mistral-7b**: QLoRA probe (25 steps or 20 min). Epochs follow the D-070 rule (`session.epoch_decision`).
- **gemma3-4b**: decided (D-078, D-079): fp32, 1 epoch, **two sessions**, no probe. Run this notebook twice.
  Session 1 finishes unharmful; harmful pauses with a checkpoint at session start + 11 h and syncs.
  Session 2 restores it, resumes harmful (≈ 1.5 h) and runs the DC-07 test.

The long run starts only if elapsed + projected hours ≤ 11.5 h (`session.session_ok`; Gemma's
two-session plan is exempt). Each runtime
decision is passed as `--set` and printed. Finished pairs sync to the private store; no text is printed.

In [ ]:
REPO = "SGupta-4/Reference-Based-Bias-Detection-Implementation"
REF = "main"  # a branch, a tag, or a FULL 40-character commit SHA (short SHAs fail)
WORK = "/kaggle/working/rbbd"
MODEL = "llama3.1-8b"  # sessions in order: llama3.1-8b, mistral-7b, gemma3-4b
CFG = f"configs/tier1_{MODEL}.yaml"
PROBE = None if MODEL == "llama3.1-8b" else f"configs/m3c_probe_{MODEL}.yaml"
import time
SESSION_T0 = time.time()

In [ ]:
# Clone the repo at REF (branch, tag or commit SHA). If the repo is private, attach the
# GITHUB_TOKEN secret: it is passed to git as an HTTP header through environment
# variables, never in the URL, argv or output.
import base64, datetime, os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

os.environ.setdefault(
    "RBBD_SESSION_ID", datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
RBBD_SESSION_ID = os.environ["RBBD_SESSION_ID"]  # also expands $RBBD_SESSION_ID in ! lines
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # fail instead of prompting for a password
try:
    _gh = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _gh = None
if _gh:
    _basic = base64.b64encode(f"x-access-token:{_gh}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {_basic}")
del _gh

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
for args in (["init", "-q"], ["remote", "add", "origin", f"https://github.com/{REPO}.git"],
             ["fetch", "-q", "--depth", "1", "origin", REF], ["checkout", "-q", "FETCH_HEAD"]):
    done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode and args[0] == "fetch" and "GIT_CONFIG_COUNT" in git_env:
        # An expired or revoked token gets HTTP 401 even on a public repo (B-025): retry
        # without it. git's stderr never contains the header value.
        print("GITHUB_TOKEN was rejected (expired or revoked?); retrying without it")
        git_env = {k: v for k, v in git_env.items() if not k.startswith("GIT_CONFIG_")}
        done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed: {done.stderr[-500:]}")
del git_env
os.chdir(WORK)
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
      "| session", os.environ["RBBD_SESSION_ID"])

In [ ]:
!pip install -q -e ".[train,dev]" 2>&1 | tail -5
!pip freeze > /kaggle/working/pip_freeze_m3c.txt
# Preflight (B-012): fails here, not inside a stage, if torch and torchvision do not match.
!python -c "import torch, torchvision, transformers; from transformers import Qwen2ForCausalLM, LlamaForCausalLM, Gemma3ForConditionalGeneration; print('preflight ok', torch.__version__, torchvision.__version__, transformers.__version__)"

In [ ]:
# Copy Kaggle Secrets into the environment (presence flags only are printed) and keep the
# HF cache on ephemeral disk so it never counts against /kaggle/working (D-029/D-041).
import sys
sys.path.insert(0, f"{WORK}/src")  # the editable install is not visible to this already-running kernel
from rbbd.utils.env import ephemeral_dir, load_kaggle_secrets

_secrets = load_kaggle_secrets(["HF_TOKEN"])
print(_secrets)
if not _secrets["HF_TOKEN"]:
    # Stop here instead of failing later on every gated download (B-022).
    raise RuntimeError("HF_TOKEN secret not attached: Add-ons -> Secrets -> enable HF_TOKEN "
                       "for this notebook, then run again")
os.environ["HF_HOME"] = str(ephemeral_dir() / "hf")
ENV_DIR = f"/kaggle/working/artifacts/env/{os.environ['RBBD_SESSION_ID']}"
os.makedirs(ENV_DIR, exist_ok=True)

In [ ]:
# Restore this model's finished jobs (done.json, finals) from the private store, then select
# the data (ftdata ids are deterministic, so train keys match the earlier session).
import json
from rbbd import config as _config
_cfg = _config.load(CFG)
SLUG = _cfg["models"][0]["slug"]
MODEL_ID = _cfg["models"][0]["id"]
!python -m rbbd.cli restore --path train/{SLUG} || echo "nothing to restore yet"
!python -m rbbd.cli run --config {CFG} --stages ftdata
!python -c "from rbbd.models.loading import LoadSpec, download; print(download(LoadSpec('{MODEL_ID}')))"

In [ ]:
# Runtime decisions (pre-registered: D-070 epochs, D-076 Gemma precision, session bound).
import glob, subprocess
from pathlib import Path
from rbbd import runner
from rbbd.finetune import session as ses, sft
from rbbd.utils import manifest as mf
ROOT = Path("/kaggle/working/artifacts")
OVERRIDES, DECISION = [], {}

def run_probe(extra=()):
    cmd = [sys.executable, "-m", "rbbd.cli", "run", "--config", PROBE, "--stages", "ftdata,train"]
    for a in extra:
        cmd += ["--set", a]
    t0 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"probe {list(extra)}: exit {r.returncode} after {time.time() - t0:.0f} s"); print(r.stdout[-600:])
    logs = ""
    if r.returncode:
        print(r.stderr[-2000:])
        for lg in sorted(glob.glob(f"{ROOT}/train/{PROBE_SLUG}/*/*/seed0/*/train.log")):
            txt = open(lg).read(); logs += txt
            print("----", lg.split("/train/")[1]); print("".join(txt.splitlines(True)[-20:]))
    return logs

if PROBE is None:
    DECISION = {"epochs": 1, "hours": 6.41, "reason": "D-071: Llama probe, 1 epoch at batch 2x16"}
elif MODEL == "gemma3-4b":
    # Decided (D-078/D-079): fp16 fails; 1 epoch in fp32 over two sessions. The config already
    # holds compute fp32 and epochs 1; the session deadline pauses and the next session resumes.
    DECISION = {"epochs": 1, "hours": None, "two_sessions": True,
                "reason": "D-079: fp32, 1 epoch, paused at the deadline and resumed next session"}
else:
    PROBE_SLUG = _config.load(PROBE)["models"][0]["slug"]
    compute = None
    if MODEL == "gemma3-4b":
        logs = run_probe()
        DECISION["fp16"] = ses.gemma_fp16_ok(ses.probe_results(ROOT, PROBE_SLUG, compute="fp16"), logs)
        compute = "fp16" if DECISION["fp16"]["ok"] else "fp32"
        if compute == "fp32":
            OVERRIDES.append("train.compute=fp32")
            run_probe(["train.compute=fp32"])
    else:
        run_probe()
    PROBE_DONE = ses.probe_results(ROOT, PROBE_SLUG, compute=compute)
    DECISION["probe"] = {s: {k: d.get(k) for k in ("tokens_per_second_steady", "peak_mem_gib",
                         "projected_hours", "global_step", "scaler_skipped_steps", "max_abs_hidden")}
                         for s, d in PROBE_DONE.items()}
    DECISION.update(ses.epoch_decision(PROBE_DONE.get("harmful")))
    if DECISION["epochs"] == 1:
        OVERRIDES.append("train.epochs=1")
ELAPSED = (time.time() - SESSION_T0) / 3600
PROCEED = DECISION.get("epochs") is not None and (
    DECISION.get("two_sessions") or ses.session_ok(ELAPSED, DECISION.get("hours")))
SET_ARGS = " ".join(f"--set {a}" for a in OVERRIDES)
print(json.dumps({"decision": DECISION, "overrides": OVERRIDES, "elapsed_h": round(ELAPSED, 2),
                  "proceed": PROCEED}, indent=1, default=str))

In [ ]:
# The long run (u||h QLoRA). Every job pauses with a checkpoint at the session deadline
# (session start + 11 h, D-079) instead of being killed; the next session resumes it.
DEADLINE_H = 11.0
os.environ["RBBD_TRAIN_DEADLINE_UNIX"] = str(SESSION_T0 + DEADLINE_H * 3600)
if PROCEED:
    !python -m rbbd.cli run --config {CFG} --stages ftdata,train {SET_ARGS} || true
    for _log in sorted(glob.glob(f"/kaggle/working/artifacts/train/{SLUG}/*/*/*/*/train.log")):
        if "Error" in open(_log).read():
            print("----", _log.split("/train/")[1]); print("".join(open(_log).readlines()[-25:]))
else:
    print("NOT RUNNING: no epoch decision or the session bound does not fit (D-070/D-076); paste the decision cell")
_eff = _config.load(CFG, OVERRIDES)
_ft = mf.read(mf.manifest_path(ROOT, "ftdata", runner.stage_run_keys(_eff)["ftdata"]))
JOBS = sft.plan_jobs(_eff, ROOT, list(_ft.output_hashes)) if _ft and _ft.complete else []
ALL_DONE = bool(JOBS) and all((j.out_dir / "done.json").exists() for j in JOBS)
PAUSED = [j.split for j in JOBS if not (j.out_dir / "done.json").exists()]
print("all endpoints done:", ALL_DONE, "| paused / unfinished:", PAUSED)

In [ ]:
# DC-07 on the real Tier 1 endpoints + an α=0.5 forward on the inference base (precision as M4).
if PROCEED and ALL_DONE:
    os.environ["RBBD_M3C_CONFIG"] = CFG
    os.environ["RBBD_M3C_SET"] = ";".join(OVERRIDES)
    !pytest -q -m gpu -rfE --tb=short tests/gpu/test_train_gpu.py -k tier1 2>&1 | tee $ENV_DIR/pytest_gpu_m3c.txt | tail -30

In [ ]:
# End-of-session sync (private store, D-041).
for _path in ["manifests", "ftdata", f"train/{SLUG}", f"env/{RBBD_SESSION_ID}"]:
    if os.path.isdir(f"/kaggle/working/artifacts/{_path}"):
        !python -m rbbd.cli sync --path {_path}

In [ ]:
# Summary to paste back (aggregates only).
print(json.dumps({"decision": DECISION, "overrides": OVERRIDES, "proceed": PROCEED,
                  "all_done": ALL_DONE, "paused": PAUSED}, default=str))
for path in sorted(glob.glob(f"/kaggle/working/artifacts/train/{SLUG}/*/*/*/*/done.json")):
    d = json.load(open(path))
    print(path.split("/train/")[1]); print(json.dumps({k: d.get(k) for k in ("global_step", "epochs",
          "train_loss", "final_loss", "tokens_per_second_steady", "peak_mem_gib", "scaler_skipped_steps",
          "max_abs_hidden", "resumed_from", "seconds_this_process")}))
for path in sorted(glob.glob(f"/kaggle/working/artifacts/train/{SLUG}/*/*/*/*/oom.json")):
    print("OOM:", path.split("/train/")[1], open(path).read()[:200])
for path in sorted(glob.glob(f"{ENV_DIR}/m3a_train_gpu.json")):
    print(open(path).read())
!df -h /kaggle/working | tail -1
!python -m rbbd.cli status --config {CFG} {SET_ARGS}